# THGS Mask Proposer Timing — SAM ViT-H vs FastSAM-x

**한 줄 목적**: 같은 GPU·같은 이미지·같은 resolution 에서 **mask 생성 런타임만** 직접 비교한다.

이 노트북은 `THGS_2D_Source_AB.ipynb` 와 달리 CLIP·평가 단계 없이 *proposer 자체의 속도* 만 본다.
측정 항목:
- 모델 load time (1회)
- per-image `t_mask_ms` (warm-up 제외)
- per-image `n_masks` (4-level 합산)
- per-image peak GPU memory
- 모델 가중치 디스크 크기

## Arm 정의 (THGS_2D_Source_AB 와 동일)
| arm | proposer | 4-scale 의미 |
|---|---|---|
| **A** | SAM ViT-H + `SamAutomaticMaskGenerator` (LangSplat fork) | 한 번의 `.generate()` 가 area 기준으로 4 list (default/s/m/l) 반환 |
| **B** | FastSAM-x (ultralytics) | conf threshold `[0.4, 0.5, 0.6, 0.7]` 로 **4번 forward** |

## 셀 구조
| 셀 | 내용 |
|---|---|
| 0 | 환경 (GPU, Drive, repo clone) |
| 1 | 의존성 설치 |
| 2 | SAM + FastSAM 가중치 다운로드 (디스크 사이즈 기록) |
| 3 | LERF-OVS ramen 이미지 확보 + 벤치 프레임 선택 |
| 4 | SAM 벤치 (load + warm-up + timed loop) |
| 5 | FastSAM 벤치 (load + warm-up + timed loop) |
| 6 | 통합 통계 표 (mean / median / p95 / std) |
| 7 | 시각화 (bar / 분포 / scatter / mem) |
| 8 | `md/proposer_timing_summary.md` 자동 생성 + Drive 백업 |

---
## Cell 0. 환경 설정

In [ ]:
!nvidia-smi -L
import torch, platform
print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")
print(f"GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")
print(f"Python : {platform.python_version()}")

from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_CACHE = "/content/drive/MyDrive/THGS_cache"
os.makedirs(DRIVE_CACHE, exist_ok=True)
print(f"Drive 캐시: {DRIVE_CACHE}")

In [ ]:
# THGS repo (mask_nms / fastsam_4scale 재사용용)
import os
REPO_URL = "https://github.com/BAEJUNHAK/THGS.git"
WORK_DIR = "/content/THGS"
if not os.path.exists(WORK_DIR):
    !git clone --recursive {REPO_URL} {WORK_DIR}
os.chdir(WORK_DIR)
!git pull --rebase --autostash 2>&1 | tail -3
print("work dir:", os.getcwd())

---
## Cell 1. 의존성

In [ ]:
!pip install -q --upgrade opencv-python
!pip install -q ultralytics tqdm pandas matplotlib seaborn gdown omegaconf
# open_clip — encode_fastsam_clip.py 가 모듈-레벨에서 import 하므로 필수
!pip install -q open_clip_torch
# segment-anything (LangSplat fork) — A 의 4-output generator
!pip install -q git+https://github.com/minghanqin/segment-anything-langsplat.git

import ultralytics, segment_anything, open_clip
print("ultralytics:", ultralytics.__version__)
print("open_clip  :", open_clip.__version__)
print("segment_anything (LangSplat fork) installed")

---
## Cell 2. 가중치 다운로드 + 디스크 크기 기록

In [ ]:
import os, shutil
os.makedirs("ckpts", exist_ok=True)

SAM_CKPT     = "ckpts/sam_vit_h_4b8939.pth"     # ~2.5 GB
FASTSAM_CKPT = "ckpts/FastSAM-x.pt"             # ~145 MB

# SAM ViT-H — Drive 우선, 없으면 facebook URL
drive_sam = f"{DRIVE_CACHE}/sam_vit_h_4b8939.pth"
if not os.path.exists(SAM_CKPT):
    if os.path.exists(drive_sam):
        print("SAM: Drive에서 복원")
        shutil.copy2(drive_sam, SAM_CKPT)
    else:
        print("SAM: facebook 원본 다운로드")
        !wget -q --show-progress -O {SAM_CKPT} \
            https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
        # Drive 캐시에 백업
        shutil.copy2(SAM_CKPT, drive_sam)

# FastSAM-x — ultralytics 자동 dl + GitHub fallback
drive_fast = f"{DRIVE_CACHE}/FastSAM-x.pt"
if not os.path.exists(FASTSAM_CKPT):
    if os.path.exists(drive_fast):
        print("FastSAM: Drive에서 복원")
        shutil.copy2(drive_fast, FASTSAM_CKPT)
    else:
        try:
            from ultralytics import FastSAM
            m = FastSAM('FastSAM-x.pt')
            for c in ['FastSAM-x.pt', getattr(m, 'ckpt_path', None)]:
                if c and os.path.exists(c):
                    shutil.copy2(c, FASTSAM_CKPT); break
            del m
        except Exception as e:
            print("ultralytics 자동 dl 실패:", e)
    if not os.path.exists(FASTSAM_CKPT):
        !wget -q --show-progress -O {FASTSAM_CKPT} \
            https://github.com/ultralytics/assets/releases/download/v8.2.0/FastSAM-x.pt
    if os.path.exists(FASTSAM_CKPT) and not os.path.exists(drive_fast):
        shutil.copy2(FASTSAM_CKPT, drive_fast)

assert os.path.exists(SAM_CKPT)     and os.path.getsize(SAM_CKPT)     > 100_000_000
assert os.path.exists(FASTSAM_CKPT) and os.path.getsize(FASTSAM_CKPT) >  10_000_000

size_sam_mb     = os.path.getsize(SAM_CKPT)     / 1024 / 1024
size_fastsam_mb = os.path.getsize(FASTSAM_CKPT) / 1024 / 1024
print(f"SAM ViT-H : {size_sam_mb:8.1f} MB  ({SAM_CKPT})")
print(f"FastSAM-x : {size_fastsam_mb:8.1f} MB  ({FASTSAM_CKPT})")
print(f"ratio (SAM / FastSAM): {size_sam_mb / size_fastsam_mb:.1f}×")

---
## Cell 3. 데이터 + 벤치 프레임 선택

ramen 씬 이미지를 사용한다. 두 모델 모두 동일한 (정렬된) N 장에 대해 측정.
`N_BENCH = 30` (warm-up 5장 포함, 통계는 25장).

In [ ]:
import os, zipfile, gdown
from omegaconf import OmegaConf

DATASET = "lerf"; SCENE = "ramen"
cfg = OmegaConf.load(f"configs/{DATASET}.yml")
DATA_PATH = cfg.dataset.data_path
IMG_DIR   = f"{DATA_PATH}/{SCENE}/images"

if not os.path.exists(IMG_DIR):
    print("LERF-OVS 다운로드 ...")
    gdown.download(id="1QF1Po5p5DwTjFHu6tnTeYs_G0egMVmHt",
                   output="/content/lerf_ovs.zip", quiet=False)
    with zipfile.ZipFile("/content/lerf_ovs.zip", 'r') as z:
        z.extractall("data/")
    os.remove("/content/lerf_ovs.zip")
    if not os.path.exists(DATA_PATH) and os.path.exists("data/lerf_ovs"):
        os.symlink("lerf_ovs", DATA_PATH)

all_imgs = sorted(f for f in os.listdir(IMG_DIR) if f.lower().endswith(('.jpg', '.png', '.jpeg')))
print(f"총 {len(all_imgs)}장")

# === 측정 파라미터 ===
N_BENCH   = 30        # 측정 프레임 수 (warm-up 포함)
WARMUP    = 5         # 통계에서 제외할 첫 N장
MAX_H     = 1080      # A 의 image_encoding.py 와 동일한 1080P cap
BENCH_FRAMES = all_imgs[:N_BENCH]

OUT_ROOT = "output/exp_proposer_timing"
os.makedirs(OUT_ROOT, exist_ok=True)

print(f"BENCH_FRAMES = {N_BENCH} frames (warm-up {WARMUP} 제외 → 통계 {N_BENCH-WARMUP}장)")
print(f"resolution cap: H ≤ {MAX_H}")
print(f"OUT_ROOT     : {OUT_ROOT}")

In [ ]:
# 공통 이미지 로더 — 두 arm 모두 동일한 input 받도록
import cv2, numpy as np

def load_image_for_bench(path, max_h=MAX_H):
    img = cv2.imread(path)
    h, w = img.shape[:2]
    if h > max_h:
        scale = max_h / h
        img = cv2.resize(img, (int(w * scale), max_h), interpolation=cv2.INTER_AREA)
    return img  # BGR HxWx3 uint8

_test = load_image_for_bench(os.path.join(IMG_DIR, BENCH_FRAMES[0]))
print(f"sample shape after cap: {_test.shape}")

---
## Cell 4. SAM ViT-H 벤치

`SamAutomaticMaskGenerator(points_per_side=32, crop_n_layers=1, …)` 은 image_encoding.py 와 동일.
LangSplat fork 의 `.generate()` 는 **한 번 호출에 4-list (default/s/m/l)** 를 반환한다.

In [ ]:
import time, gc, torch, pandas as pd
from tqdm import tqdm
from segment_anything import SamAutomaticMaskGenerator, sam_model_registry

torch.cuda.empty_cache(); gc.collect()
torch.cuda.reset_peak_memory_stats()
vram_before_mb = torch.cuda.memory_allocated() / 1024 / 1024

# === Load ===
torch.cuda.synchronize(); t0 = time.perf_counter()
sam = sam_model_registry["vit_h"](checkpoint=SAM_CKPT).to('cuda')
mask_generator = SamAutomaticMaskGenerator(
    model=sam,
    points_per_side=32,
    pred_iou_thresh=0.7,
    box_nms_thresh=0.7,
    stability_score_thresh=0.85,
    crop_n_layers=1,
    crop_n_points_downscale_factor=1,
    min_mask_region_area=100,
)
torch.cuda.synchronize()
sam_load_ms = (time.perf_counter() - t0) * 1000
vram_after_load_mb = torch.cuda.memory_allocated() / 1024 / 1024

n_params = sum(p.numel() for p in sam.parameters())
print(f"SAM ViT-H load: {sam_load_ms/1000:.2f} s | params: {n_params/1e6:.1f} M | VRAM after load: {vram_after_load_mb:.0f} MB")

# === Timed loop ===
records = []
for i, fname in enumerate(tqdm(BENCH_FRAMES, desc="SAM")):
    img_bgr = load_image_for_bench(os.path.join(IMG_DIR, fname))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    masks_d, masks_s, masks_m, masks_l = mask_generator.generate(img_rgb)
    torch.cuda.synchronize()
    dt_ms = (time.perf_counter() - t0) * 1000

    records.append({
        'frame': fname, 'is_warmup': i < WARMUP,
        't_mask_ms': dt_ms,
        'n_d': len(masks_d), 'n_s': len(masks_s),
        'n_m': len(masks_m), 'n_l': len(masks_l),
        'n_masks': len(masks_d) + len(masks_s) + len(masks_m) + len(masks_l),
        'peak_mem_mb': torch.cuda.max_memory_allocated() / 1024 / 1024,
        'H': img_bgr.shape[0], 'W': img_bgr.shape[1],
    })

df_sam = pd.DataFrame(records)
df_sam['model'] = 'SAM_ViTH_4scale'
df_sam['ckpt_size_mb'] = size_sam_mb
df_sam['load_ms']      = sam_load_ms
df_sam['vram_load_mb'] = vram_after_load_mb
df_sam['n_params_M']   = n_params / 1e6
df_sam.to_csv(f"{OUT_ROOT}/timing_sam.csv", index=False)
print(f"\nsaved: {OUT_ROOT}/timing_sam.csv")
print(df_sam[['frame','is_warmup','t_mask_ms','n_masks','peak_mem_mb']].head(8).to_string(index=False))

# === Free ===
del sam, mask_generator
torch.cuda.empty_cache(); gc.collect()
print(f"\nVRAM after free: {torch.cuda.memory_allocated()/1024/1024:.0f} MB")

---
## Cell 5. FastSAM-x 벤치

`encode_fastsam_clip.fastsam_4scale` 와 같은 conf 4-pass 방식. 각 이미지에 대해 4번 forward 의 합을 `t_mask_ms` 로 기록한다.

In [ ]:
import sys; sys.path.insert(0, 'scripts')
from encode_fastsam_clip import fastsam_4scale, LEVEL_NAMES
from ultralytics import FastSAM
import time, gc, torch, numpy as np, pandas as pd
from tqdm import tqdm

torch.cuda.empty_cache(); gc.collect()
torch.cuda.reset_peak_memory_stats()

# === Load (weights only — first-forward JIT cost lands in is_warmup=True frames, parity with SAM) ===
torch.cuda.synchronize(); t0 = time.perf_counter()
fast_model = FastSAM(FASTSAM_CKPT)
torch.cuda.synchronize()
fast_load_ms = (time.perf_counter() - t0) * 1000
vram_after_load_mb = torch.cuda.memory_allocated() / 1024 / 1024

# n_params (ultralytics 모델 내부 — 빌드 직후엔 .model.model 이 비어있을 수 있어 fallback)
try:
    n_params = sum(p.numel() for p in fast_model.model.model.parameters())
    if n_params == 0: raise AttributeError
except (AttributeError, TypeError):
    try:
        n_params = sum(p.numel() for p in fast_model.model.parameters())
    except Exception:
        n_params = 0
print(f"FastSAM-x load (weights only): {fast_load_ms/1000:.2f} s | params: {n_params/1e6:.1f} M | VRAM after load: {vram_after_load_mb:.0f} MB")
print("  (CUDA JIT cost will be absorbed by the first {} warm-up frames)".format(WARMUP))

# === Timed loop ===
records = []
for i, fname in enumerate(tqdm(BENCH_FRAMES, desc="FastSAM")):
    img_bgr = load_image_for_bench(os.path.join(IMG_DIR, fname))

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    masks_per_level = fastsam_4scale(fast_model, img_bgr)   # 4 forward 합산
    torch.cuda.synchronize()
    dt_ms = (time.perf_counter() - t0) * 1000

    n = {k: len(v) for k, v in masks_per_level.items()}
    records.append({
        'frame': fname, 'is_warmup': i < WARMUP,
        't_mask_ms': dt_ms,
        'n_d': n['default'], 'n_s': n['s'],
        'n_m': n['m'], 'n_l': n['l'],
        'n_masks': sum(n.values()),
        'peak_mem_mb': torch.cuda.max_memory_allocated() / 1024 / 1024,
        'H': img_bgr.shape[0], 'W': img_bgr.shape[1],
    })

df_fast = pd.DataFrame(records)
df_fast['model'] = 'FastSAM_x_4conf'
df_fast['ckpt_size_mb'] = size_fastsam_mb
df_fast['load_ms']      = fast_load_ms
df_fast['vram_load_mb'] = vram_after_load_mb
df_fast['n_params_M']   = n_params / 1e6
df_fast.to_csv(f"{OUT_ROOT}/timing_fastsam.csv", index=False)
print(f"\nsaved: {OUT_ROOT}/timing_fastsam.csv")
print(df_fast[['frame','is_warmup','t_mask_ms','n_masks','peak_mem_mb']].head(8).to_string(index=False))

del fast_model
torch.cuda.empty_cache(); gc.collect()

---
## Cell 6. 통합 통계 표

warm-up 제외한 후 mean / median / p95 / std / min / max.

In [ ]:
import pandas as pd, numpy as np

df_sam  = pd.read_csv(f"{OUT_ROOT}/timing_sam.csv")
df_fast = pd.read_csv(f"{OUT_ROOT}/timing_fastsam.csv")
df_all  = pd.concat([df_sam, df_fast], ignore_index=True)
# is_warmup 이 'True'/'False' 문자열로 읽힐 수 있어 명시적 bool 캐스팅
df_all['is_warmup'] = df_all['is_warmup'].astype(str).str.lower().eq('true')
df_eval = df_all[~df_all['is_warmup']].copy()

def summarize(g):
    return pd.Series({
        'n_frames'        : len(g),
        't_mask_mean_ms'  : g['t_mask_ms'].mean(),
        't_mask_median_ms': g['t_mask_ms'].median(),
        't_mask_p95_ms'   : g['t_mask_ms'].quantile(0.95),
        't_mask_std_ms'   : g['t_mask_ms'].std(),
        't_mask_min_ms'   : g['t_mask_ms'].min(),
        't_mask_max_ms'   : g['t_mask_ms'].max(),
        'n_masks_mean'    : g['n_masks'].mean(),
        'peak_mem_mb_mean': g['peak_mem_mb'].mean(),
        'load_ms'         : g['load_ms'].iloc[0],
        'ckpt_size_mb'    : g['ckpt_size_mb'].iloc[0],
        'n_params_M'      : g['n_params_M'].iloc[0],
    })

# pandas 1.x / 2.x 양립 — include_groups 미지정
import warnings
with warnings.catch_warnings():
    warnings.simplefilter('ignore', FutureWarning)
    summary = df_eval.groupby('model', group_keys=False).apply(summarize).round(2)
print("=== Per-image timing (warm-up 제외) ===")
print(summary.T.to_string())

# Speedup ratio
row_sam  = summary.loc['SAM_ViTH_4scale']
row_fast = summary.loc['FastSAM_x_4conf']
speedup_mean   = row_sam['t_mask_mean_ms']   / row_fast['t_mask_mean_ms']
speedup_median = row_sam['t_mask_median_ms'] / row_fast['t_mask_median_ms']
load_speedup   = row_sam['load_ms']          / row_fast['load_ms']
size_ratio     = row_sam['ckpt_size_mb']     / row_fast['ckpt_size_mb']
param_ratio    = row_sam['n_params_M']       / row_fast['n_params_M']
mem_ratio      = row_sam['peak_mem_mb_mean'] / row_fast['peak_mem_mb_mean']

print("\n=== Speedup (B 가 A 대비 몇 배 빠른가) ===")
print(f"  per-image (mean)   : {speedup_mean:6.1f} ×")
print(f"  per-image (median) : {speedup_median:6.1f} ×")
print(f"  load time          : {load_speedup:6.1f} ×")
print(f"  ckpt disk size     : {size_ratio:6.1f} × smaller (B)")
print(f"  params             : {param_ratio:6.1f} × fewer (B)")
print(f"  peak GPU mem       : {mem_ratio:6.1f} × less   (B)")

summary.to_csv(f"{OUT_ROOT}/summary_table.csv")

---
## Cell 7. 시각화

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# (1) Bar: mean t_mask with std
ax = axes[0, 0]
models = ['SAM_ViTH_4scale', 'FastSAM_x_4conf']
means  = [row_sam['t_mask_mean_ms'], row_fast['t_mask_mean_ms']]
stds   = [row_sam['t_mask_std_ms'],  row_fast['t_mask_std_ms']]
colors = ['#4C72B0', '#DD8452']
ax.bar(models, means, yerr=stds, color=colors, capsize=8)
for i, (m, s) in enumerate(zip(means, stds)):
    ax.text(i, m + s + max(means)*0.02, f'{m:,.0f} ± {s:,.0f} ms',
            ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('t_mask (ms / image)')
ax.set_title(f'Mean mask-generation time\n(speedup ≈ {speedup_mean:.1f}×)')
ax.set_yscale('log')

# (2) Distribution histogram (log x)
ax = axes[0, 1]
for m, c in zip(models, colors):
    d = df_eval[df_eval['model'] == m]['t_mask_ms']
    ax.hist(d, bins=15, alpha=0.6, label=f"{m} (n={len(d)})", color=c)
ax.set_xscale('log')
ax.set_xlabel('t_mask (ms, log)'); ax.set_ylabel('frames')
ax.set_title('Per-image distribution')
ax.legend(fontsize=8)

# (3) Per-frame line (consistency)
ax = axes[0, 2]
for m, c in zip(models, colors):
    d = df_all[df_all['model'] == m].reset_index(drop=True)
    ax.plot(d.index, d['t_mask_ms'], 'o-', color=c, label=m, markersize=4)
    ax.axvspan(0, WARMUP - 0.5, alpha=0.1, color='gray')
ax.set_yscale('log')
ax.set_xlabel('frame index (gray = warm-up)')
ax.set_ylabel('t_mask (ms, log)')
ax.set_title('Per-image consistency')
ax.legend(fontsize=8)

# (4) Scatter: n_masks vs t_mask
ax = axes[1, 0]
for m, c in zip(models, colors):
    d = df_eval[df_eval['model'] == m]
    ax.scatter(d['n_masks'], d['t_mask_ms'], color=c, label=m, s=40, alpha=0.7)
ax.set_xlabel('n_masks (4-level total)')
ax.set_ylabel('t_mask (ms)')
ax.set_yscale('log')
ax.set_title('Cost vs mask count')
ax.legend(fontsize=8)

# (5) Peak GPU memory
ax = axes[1, 1]
mem_means = [row_sam['peak_mem_mb_mean'], row_fast['peak_mem_mb_mean']]
ax.bar(models, mem_means, color=colors)
for i, v in enumerate(mem_means):
    ax.text(i, v + max(mem_means)*0.02, f'{v:,.0f} MB',
            ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('peak GPU memory (MB)')
ax.set_title(f'Peak VRAM during inference\n(SAM uses {mem_ratio:.1f}× more)')

# (6) Disk size + load time + params
ax = axes[1, 2]
labels = ['ckpt size (MB)', 'load time (s)', 'params (M)']
sam_vals  = [row_sam['ckpt_size_mb'],  row_sam['load_ms']/1000,  row_sam['n_params_M']]
fast_vals = [row_fast['ckpt_size_mb'], row_fast['load_ms']/1000, row_fast['n_params_M']]
x = np.arange(len(labels)); w = 0.35
ax.bar(x - w/2, sam_vals,  w, label='SAM ViT-H', color=colors[0])
ax.bar(x + w/2, fast_vals, w, label='FastSAM-x', color=colors[1])
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_yscale('log')
ax.set_title('Model footprint (log scale)')
ax.legend(fontsize=8)
for i, (a, b) in enumerate(zip(sam_vals, fast_vals)):
    ax.text(i - w/2, a*1.1, f'{a:.1f}', ha='center', fontsize=8)
    ax.text(i + w/2, b*1.1, f'{b:.1f}', ha='center', fontsize=8)

plt.tight_layout()
out_p = f"{OUT_ROOT}/timing_summary.png"
plt.savefig(out_p, dpi=130, bbox_inches='tight')
plt.show()
print(f"saved: {out_p}")

---
## Cell 8. summary md 자동 생성 + Drive 백업

In [ ]:
import os, shutil, datetime

MD_DIR = "md"
os.makedirs(MD_DIR, exist_ok=True)
md_path = f"{MD_DIR}/proposer_timing_summary.md"

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
today    = datetime.date.today().isoformat()

lines = []
lines.append(f"# Mask Proposer Timing — SAM vs FastSAM\n\n")
lines.append(f"_생성일: {today} / GPU: {gpu_name} / scene: ramen / N={N_BENCH-WARMUP} (warm-up {WARMUP} 제외) / H≤{MAX_H}_\n\n")
lines.append("## Setup\n")
lines.append("- A = SAM ViT-H + `SamAutomaticMaskGenerator(points_per_side=32, crop_n_layers=1, …)`\n")
lines.append("- B = FastSAM-x + 4 forward (conf ∈ {0.4, 0.5, 0.6, 0.7})\n")
lines.append("- 두 arm 모두 같은 이미지·같은 H 캡·같은 GPU. CLIP / NMS 시간은 측정에서 제외.\n\n")

lines.append("## Per-image mask-generation timing\n\n")
lines.append("| metric | SAM ViT-H 4-scale | FastSAM-x 4-conf | ratio (SAM / FastSAM) |\n")
lines.append("|---|---:|---:|---:|\n")
for k, label in [
    ('t_mask_mean_ms',   't_mask mean (ms)'),
    ('t_mask_median_ms', 't_mask median (ms)'),
    ('t_mask_p95_ms',    't_mask p95 (ms)'),
    ('t_mask_std_ms',    't_mask std (ms)'),
    ('t_mask_min_ms',    't_mask min (ms)'),
    ('t_mask_max_ms',    't_mask max (ms)'),
    ('n_masks_mean',     'n_masks (4-level sum)'),
    ('peak_mem_mb_mean', 'peak GPU mem (MB)'),
]:
    a = row_sam[k]; b = row_fast[k]
    r = a / b if b > 0 else float('inf')
    lines.append(f"| {label} | {a:,.1f} | {b:,.1f} | {r:.2f}× |\n")

lines.append("\n## Model footprint\n\n")
lines.append("| metric | SAM ViT-H | FastSAM-x | ratio |\n|---|---:|---:|---:|\n")
for k, label in [
    ('ckpt_size_mb', 'ckpt size (MB)'),
    ('load_ms',      'load time (ms)'),
    ('n_params_M',   'params (M)'),
]:
    a = row_sam[k]; b = row_fast[k]
    r = a / b if b > 0 else float('inf')
    lines.append(f"| {label} | {a:,.1f} | {b:,.1f} | {r:.2f}× |\n")

lines.append("\n## Headline\n\n")
lines.append(f"- **Per-image speedup**: FastSAM 가 평균 **{speedup_mean:.1f}×**, median **{speedup_median:.1f}×** 빠름\n")
lines.append(f"- **Load time speedup**: **{load_speedup:.1f}×**\n")
lines.append(f"- **Disk footprint**: SAM 이 **{size_ratio:.1f}× 큼** ({row_sam['ckpt_size_mb']:,.0f} MB vs {row_fast['ckpt_size_mb']:,.0f} MB)\n")
lines.append(f"- **Peak VRAM**: SAM 이 **{mem_ratio:.1f}× 더 씀** ({row_sam['peak_mem_mb_mean']:,.0f} MB vs {row_fast['peak_mem_mb_mean']:,.0f} MB)\n")
lines.append("\n## Caveats\n\n")
lines.append("- A 의 4-scale 은 **한 번의 `.generate()` 내부 area 분할**, B 의 4-scale 은 **4 forward**. 알고리즘적으로 의미가 다른데 둘 다 \"4-level mask 출력\" 이라는 동일 contract 를 만족시키는 비용이라 같은 무대에 올렸다.\n")
lines.append("- 측정은 mask 생성만. CLIP per-mask 인코딩(arm 공통)은 제외.\n")
lines.append("- ramen 씬 단일 결과. 다른 씬에서 mask 수가 크게 달라지면 절대값은 달라질 수 있다.\n")

with open(md_path, 'w') as f:
    f.writelines(lines)
print(''.join(lines))
print(f"\nsaved: {md_path}")

# Drive 백업
drive_out = f"{DRIVE_CACHE}/{SCENE}_exp_proposer_timing"
if os.path.exists(drive_out):
    shutil.rmtree(drive_out)
shutil.copytree(OUT_ROOT, drive_out)
shutil.copy2(md_path, f"{DRIVE_CACHE}/proposer_timing_summary.md")
print(f"backed up to {drive_out} + {DRIVE_CACHE}/proposer_timing_summary.md")